In [4]:
using DifferentialEquations
using Plots
include("Chemkin.jl")

# Initial species concentrations 0. T 1. N2, 2. O2, 3. CH4, 4. H2O, 5. CO2

species_names = ["N2", "O2", "CH4", "H2O", "CO2"]
n_s = 5

X0 = zeros(length(species_names)+1)
X0[1] = 900 # K
X0[2:end] = 20.06*[7.57/10.57, 2/10.57, 1/10.57, 0, 0] #mol/m^3

# Stoichiometry: 2 O2 + CH4 -> 2 H2O + CO2
S = [0, -2, -1, 2, 1]  # N2 does not participate in the reaction
n_r = 1

species_dict = load_chemical_data("CHEMKIN-THERMDAT.txt")

Dict{String, Tuple{Float64, Vector{Float64}}} with 550 entries:
  "HSIC"          => (1500.0, [5.84954, 0.000762835, -9.97413e-8, -3.81159e-11,…
  "H2S"           => (1000.0, [2.88315, 0.00382783, -1.4234e-6, 2.498e-10, -1.6…
  "SIF3NHSIH3"    => (1000.0, [16.6994, 0.00778978, -8.11057e-7, -7.6502e-10, 1…
  "CLSI(CH3)2CH2" => (1500.0, [21.151, 0.00801827, -7.92425e-7, -3.29505e-10, 5…
  "GEF2"          => (1000.0, [4.76795, 0.00841094, -1.67642e-5, 1.56225e-8, -5…
  "CSICL3"        => (1500.0, [12.5054, 0.000533922, -2.58861e-7, 6.07531e-11, …
  "H2GAME"        => (600.0, [5.8316, 0.0122287, 3.03367e-7, -3.95694e-9, 1.225…
  "O2-"           => (1000.0, [3.88301, 0.000740787, -2.96178e-7, 5.7243e-11, -…
  "ASALME"        => (600.0, [7.12711, 0.00735786, 2.3008e-8, -2.2264e-9, 6.927…
  "CH2CLCHCL2"    => (1500.0, [16.1874, 0.00304768, -5.0115e-7, -1.5967e-11, 7.…
  "CHCLCCLOH"     => (1500.0, [14.1221, 0.00258376, -4.5769e-7, 5.21568e-12, 3.…
  "H2SI(CH3)CH2"  => (1500.0, [13.8883, 0.007

In [5]:
function Arrhenius(Y)
    T = Y[1]
    X = Y[2:end]
    # Constants from Westbrook & Dryer 1981
    n_CH4 = -0.3
    n_O2 = 1.3
    A = 1.3e8 * 1e6
    Ea = 48.4e3 # kcal/mol
    R = 1.987 # cal/mol/K
    
    # Ensure positive concentrations for exponentiation
    CH4_concentration = max(X[3], 0)
    O2_concentration = max(X[2], 0)
    
    k = A * exp(-Ea / (R * T)) # rate constant
    r = k * CH4_concentration^n_CH4 * O2_concentration^n_O2 # reaction rate
    return r
end

Arrhenius (generic function with 1 method)

In [6]:
# Temperature rate of change due to reaction enthalpy
function dT(X, r)
    h0_vec = []
    cp_vec = []
    dH = sum( S' * [h0(X[1],species_dict[species_names[i]]) for i in 1:n_s]) * r
    c_p = sum(X[2:end] .* [species_cp(X[1],species_dict[species_names[i]]) for i in 1:n_s])
    dT = -dH/c_p #J/mols * molK/J
    return dT[1]
end


dT (generic function with 1 method)

In [7]:
# Derivative function for the ODE system
function f!(dX, X, p, t)
    r = Arrhenius(X)
    dX[1] = dT(X,r)
    dX[2:end] .= r .* S  # Species concentrations change
end

# define timespan
tend = 1
tspan = (0, tend)

# define problem 
problem = ODEProblem(f!, X0, tspan)

# solve problem 
@time sol = solve(problem, alg_hints=[:stiff], abstol = 1e-6, reltol = 1e-4)

function relaxation_time(sol)
    p = 5 / 100  # convert to fraction
    final = sol.u[end]

    for (i, u) in enumerate(sol.u)
        # relative error per component
        rel_error = abs.((u .- final) ./ (final .+ eps()))  # avoid divide-by-zero
        if all(rel_error .< p)
            return sol.t[i]
        end
    end
    return NaN  # Not within threshold
end


τ = relaxation_time(sol)
println("Relaxation time: $τ seconds")

  3.898972 seconds (6.19 M allocations: 323.546 MiB, 2.29% gc time, 99.09% compilation time)
Relaxation time: 1.0 seconds


In [8]:
# Extract the solution arrays
t = sol.t
T = sol[1, :]
C = sol[2:end, :]'

println(maximum(T))

# Create a plot for the temperature
plot1 = plot(t, T, xlabel="Time (s)", ylabel="Temperature (K)")

# Create a plot for the species concentrations
plot2 = plot(t, C, xlabel="Time (s)", ylabel="Concentration", label=["N2" "O2" "CH4" "H2O" "CO2"])

plot(plot1, plot2, layout = (2,1))

savefig("1stepthermo")

901.2083926734118


"C:\\Users\\jelte\\chemicalcombustion\\Toy Models\\1stepthermo.png"